# Advection Equation (1D)

## Introduction

So far, we have considered the energy conservation equation only without material transport (advection). However, in many problems the material is moving and certain quantities, such as temperature, density, etc., must be transported (e.g., in mantle plumes). In general, mantle convection is a good example of a system in which temperature is transported both by **diffusion** (mainly in boundary layers) and by **advection** (mainly in the interior).

In the following, we consider only **advection** (i.e., $k$, $\kappa = 0$) for a one-dimensional problem. In the 1D case, the energy conservation equation reduces to the pure advection equation:

$\begin{equation}
\frac{\partial{T}}{\partial{t}}=-v_x\frac{\partial{T}}{\partial{x}}.
\end{equation}$

This equation can be solved numerically using different discretization schemes. Contrary to intuition, an FTCS discretization of the advection equation is always unstable (as can be shown by a Von Neumann stability analysis).

Here, we will implement several schemes and apply them to a specific advection problem.


## The Problem

To do this, we consider two 1-D problems (e.g., a horizontal temperature profile) with specific anomalies:

  • A Gaussian temperature distribution (a smooth transition):<br> <img src="./Figures/Exercise06_gaussian.png" alt="drawing" width="400"/> <br>

  • A block-shaped temperature anomaly (a very sharp transition):<br> <img src="./Figures/Exercise06_block.png" alt="drawing" width="400"/> <br>

We assume a spatially and temporally constant horizontal velocity $v_x$ that advects the anomaly laterally.


## The Solution

### Discretization

Despite its simple form, the advection equation is numerically relatively difficult to solve. For this purpose, we subdivide our 1D profile again into a numerical grid, where the temperature is defined at the cell centers (centroids). Since we assume that the velocity is spatially and temporally constant, no staggered grid is required.

The global indexing of the central reference point $I^C$ used in the advection equation follows the indexing described in the section on the [general solution](https://geosci-ffm.github.io/GeoModBox.jl/dev/man/GESolution). The indices of the adjacent points are defined as:

$\begin{equation}\begin{split}
I^\textrm{W} & = I^\textrm{C} - 1,\\
I^\textrm{E} & = I^\textrm{C} + 1,
\end{split}\end{equation}$

where $I$ denotes the equation number, which corresponds to the local index $i$ at the central position of the three-point stencil ($C$). $I^\textrm{W}$ and $I^\textrm{E}$ indicate the neighboring points to the west and east, respectively. These indices define the three-point stencil used in the discretized finite-difference equations below.

#### **Forward in Time and Centered in Space (FTCS)**

Let us start with the seemingly simplest approach. Approximating the partial derivatives using an *FTCS* scheme yields

$\begin{equation}
\frac{T_{I^C}^{n+1}-T_{I^C}^{n}}{\Delta{t}} = -v_x\left(\frac{T_{I^E}^{n}-T_{I^W}^{n}}{2\Delta{x}}\right),
\end{equation}$

where $\Delta{t}$ and $\Delta{x}$ denote the time step and grid spacing, respectively, $I^C$ is the central reference point, and $n$ is the temporal index. This scheme is first-order accurate in time and second-order accurate in space.

Rearranging gives the temperature at the next time step:

$\begin{equation}
T_{I^C}^{n+1} = T_{I^C}^n - v_x \Delta{t}\frac{T_{I^E}^n - T_{I^W}^n}{2\Delta{x}}.
\end{equation}$

The right-hand side can be simplified using the so-called *Courant number*:

$\begin{equation}
\alpha = \frac{v_x\Delta{t}}{\Delta{x}},
\end{equation}$

which indicates how many grid spacings a signal travels within one time step (Courant–Friedrichs–Lewy criterion, CFL). Unfortunately, this scheme is unconditionally unstable for the advection equation, as shown by a Von Neumann or Hirt stability analysis. The central differencing at $I^C$ leads to a systematic amplification of the variable (here: temperature) at each time step. The solution therefore grows continuously and is unstable.

#### **Lax–Friedrichs Method**

One way to reduce the instability of the FTCS scheme is the *Lax–Friedrichs* method. Here, the term $T_{I^C}^{n}$ is replaced by its spatial average at the same time level, which yields

$\begin{equation}
\frac{T_{I^C}^{n+1}-\left(T_{I^E}^{n}+T_{I^W}^{n}\right)/2}{\Delta{t}}=-v_x\frac{T_{I^E}^{n}-T_{I^W}^{n}}{2\Delta{x}}.
\end{equation}$

Rearranging gives:

$\begin{equation}
T_{I^C}^{n+1} = \frac{1}{2}\left(T_{I^E}^{n}+T_{I^W}^{n}\right)-
\frac{v_x \Delta{t}}{2\Delta{x}} \left(T_{I^E}^{n}-T_{I^W}^{n}\right).
\end{equation}$

This method is stable for $\alpha < 1$, but introduces significant numerical diffusion.

#### **Upwind**

Another approach is to consider only upstream information. The *upwind* scheme uses one-sided finite differences that are always taken in the upstream direction. It is first-order accurate in both space and time. The discretized advection equation becomes:

$\begin{equation}
\frac{T_{I^C}^{n+1}-T_{I^C}^n}{\Delta{t}} = -v_{x,I^C}
\begin{cases}
\frac{T_{I^C}^{n}-T_{I^W}^{n}}{\Delta{x}} &\text{if } v_{x,I^C} \gt 0
\frac{T_{I^E}^{n}-T_{I^C}^{n}}{\Delta{x}}&\text{if } v_{x,I^C} \lt 0
\end{cases}.
\end{equation}$

The scheme is stable provided the CFL criterion is satisfied ($\alpha \le 1$). Nevertheless, numerical diffusion occurs, whose strength depends on the grid spacing. A Taylor expansion shows that the scheme becomes non-diffusive in 1D with constant velocity if the CFL criterion is satisfied exactly. If it is violated, the method becomes unstable.

#### **Staggered Leapfrog Scheme**

All explicit schemes discussed so far are only first-order accurate in time and second-order accurate in space; the upwind scheme is first-order in both. To balance temporal and spatial accuracy without having to choose a very small time step, one may use the *staggered leapfrog* scheme:

$\begin{equation}
\frac{T_{I^C}^{n+1}-T_{I^C}^{n-1}}{2\Delta{t}}=-v_x\frac{T_{I^E}^{n}-T_{I^W}^{n}}{2\Delta{x}}.
\end{equation}$

This method avoids numerical diffusion, but tends to become unstable when strong gradients are present in the advected field.

#### **Semi-Lagrangian Method**

The methods discussed above each have specific drawbacks. The *semi-Lagrangian* method addresses several of them: it is stable, not constrained by the CFL criterion, and exhibits only minor numerical diffusion. It is related to tracer-based advection schemes and solves ordinary differential equations (ODEs) rather than directly applying traditional finite differences. Although it is not strictly conservative and interpolation errors can occur, it offers high accuracy and efficiency.

The central idea is to trace an advected particle backward in time to its point of origin and interpolate the corresponding value from the Eulerian grid.

In 1D, assuming a spatially and temporally constant velocity, the procedure is as follows:

**1. Compute the departure point**

The departure point $X_{i}$ of a particle that arrives at the Eulerian grid point $x_{I^C}$ at time step ${n+1}$ is given by:

$\begin{equation}
X_{i}=x_{I^C}-\Delta{t}\cdot v_{x,I^C}^{n+1},
\end{equation}$

where $x_{I^C}$ is the coordinate of the grid point $I^C$, $\Delta{t}$ is the time step, $v_x$ is the velocity in $x$ direction, and $n+1$ denotes the new time step.

**2. Interpolate the temperature**

Interpolate the temperature at time step $n$ from the neighboring Eulerian grid points onto the position $X_{i}$, for example using `cubic_spline_interpolation()`.

**3. Update the temperature field**

Assuming that the temperature at grid point $I^C$ at time step $n+1$ equals the interpolated value at $X_{i}$ at time step $n$ yields:

$\begin{equation}
T_{I^C}^{n+1} = T_{X_{I^C}}^n
\end{equation}$

#### **Passive Markers**

Using passive markers is a fully Lagrangian method. Markers distributed initially are transported by the prescribed velocity field and can carry different physical properties. If these properties influence the rheology or dynamics of the model, they are referred to as *active markers*, which requires an adjustment of the velocity field.

To transport a property with markers, the following steps are required:

**1. Define the marker vector** $\vec{x}_p$ with initial positions $\vec{x}_p\left(t=0\right)$ and initial property values $f\left(\vec{x}_p\left(t=0\right)\right)$.

**2. Compute flow paths** by solving the ODE of particle motion, for example using Forward Euler or Runge–Kutta integration.

**3. Interpolate grid values** $f(x,t)$ at the marker positions $\vec{x}_p$, e.g., using bilinear interpolation.

Further information on the marker routine can be found in the [documentation](https://geosci-ffm.github.io/GeoModBox.jl/dev/man/AdvOneD).

In the following, we will implement the different advection schemes and advect a 1D temperature anomaly laterally.


To solve the problem using Julia, we first need to define the required modules (`Plots, Interpolations`) and submodules (`GeoModBox.HeatEquation.TwoD, GeoModBox.Tracers.OneD`):

In [ ]:
using Plots, Interpolations
using ?

Let us now define our geometry and the required numerical parameters (i.e., grid resolution, grid, etc.):

In [ ]:
# Geometric constants --------------------------------------------------- #
xmin    =   ?                           # [m]
xmax    =   ?                          # [m]
# ----------------------------------------------------------------------- #
# Numerical constants --------------------------------------------------- #
nc      =   ?                         # Number of grid points
Δx      =   ?                     # Grid spacing
# ---
xc      =   ?   # Cell-centered x-coordinate
xce     =   ?   # Staggered x-coordinate (with ghost cells)
X       =   zeros(nc)
# ----------------------------------------------------------------------- #
# Maximum simulation time ----------------------------------------------- #
tmax    =   ?                        # [s]
# ----------------------------------------------------------------------- #
# Horizontal velocity --------------------------------------------------- #
vx      =   ?                         # [m/s]
# ----------------------------------------------------------------------- #

Now we can use the CFL criterion to define the maximum time step size:

In [ ]:
# Time-step definition -------------------------------------------------- #
Δtfac   =   ?                         # Courant factor
Δt      =   ?        # Time-step size
nt      =   ceil(Int, tmax/Δt)          # Number of time steps
# ----------------------------------------------------------------------- #

We now define a variable to select the finite-difference (FD) method for advecting the temperature:

* `FTCS`     – forward in time, centered in space
* `upwind`   – upwind scheme
* `lax`      – Lax–Friedrichs scheme
* `slf`      – staggered leapfrog scheme
* `semilag`  – semi-Lagrangian scheme
* `tracers`  – passive marker method

In addition, we define the initial profile via:

* `block`     – block-shaped anomaly
* `gaussian`  – Gaussian anomaly

The variables are stored in the corresponding tuples (`FD`, `Ini`).


In [ ]:
FD          =   (Method     = (Adv=:upwind,),)
Ini         =   (T=:gaussian,)

To visualize and save the animation file, the output directory must be specified:

In [ ]:
# Animationssettings ---------------------------------------------------- #
path        =   string("./Results/")
anim        =   Plots.Animation(path, String[] )
filename    =   string("06_1D_advection_",Ini.T,"_",FD.Method.Adv)
save_fig    =   1
# ----------------------------------------------------------------------- #

If we want to use the tracer method, we must additionally specify the number of tracers per grid length:

In [ ]:
# Tracer advection method ----------------------------------------------- #
nmx         =   ?       #   Number of tracers per "cell"
# ----------------------------------------------------------------------- #

Let us now define and plot the initial condition (i.e., the initial temperature profile):

In [ ]:
if Ini.T == :block 
    # Background temperature -------------------------------------------- #
    Tb      =   ?                # [K]
    
    # Location and intensity of the temperature anomaly ----------------- #
    xTl     =   ?
    xTr     =   ?
    Ta      =   ?                # [K]
    
    # Construct initial temperature profile ----------------------------- #
    T       =   ?
    T[ ? ]    .=  ?
    Tmin    =   minimum(T)
    Tmax    =   maximum(T)
    tc      =   100
elseif Ini.T == :gaussian
    # Gaussian temperature distribution -------------------------------- #
    Tb      =   ?                # Background temperature
    Ampl    =   ?                 # Amplitude
    sigma   =   ?                   # Standard deviation
    xcG     =   ?  # x-coordinate of the maximum
    T       =   zeros(nc)
    @. T    =   Tb + Ampl * exp(-((xc - xcG)^2) / sigma^2)
    
    Tmin    =   minimum(T)
    Tmax    =   maximum(T)
    tc      =   Ampl
end

TWE             =   zeros(nc + 2)
TWE[2:end-1]    .=  T
TWE[1]          =   T[end]
TWE[end]        =   T[1]
TWE2            =   zeros(nc + 2)

q = plot( ? )
if save_fig == 0
    display(q)
end

if FD.Method.Adv == :tracers
    # Total number of tracers ------------------------------------------- #
    nm          =   (nc) * nmx
    # Tracer spacing ----------------------------------------------------- #
    Δmx         =   (abs(xmin) + abs(xmax)) / (nm + 1)
    # x-coordinates of tracers ------------------------------------------ #
    xm          =   collect(xmin + Δmx : Δmx : xmax - Δmx) .+ rand(nm) .* 0.5 * Δmx    
    # Tracer temperatures ------------------------------------------------ #
    Tm          =   zeros(nm)    
end

We can now implement the equations to solve the advection equation.

In [ ]:
# Solving the advection equation --------------------------------------- #
for i = 2:nt
    display(string("Time step: ", i))

    # Periodic ghost cells (west/east)
    TWE[2:end-1] .= T
    TWE[1]       =  T[end]
    TWE[end]     =  T[1]
    
    if FD.Method.Adv == :FTCS
        # Forward Time, Centered Space (unstable for pure advection)
        T .= ?

    elseif FD.Method.Adv == :upwind
        # Upwind (first-order, direction depends on sign of vx)
        if vx > 0
            T .= ?
        elseif vx < 0
            T .= ?
        end

    elseif FD.Method.Adv == :downwind
        # Downwind (anti-diffusive, generally unstable)
        T .= ?

    elseif FD.Method.Adv == :lax
        # Lax (Lax–Friedrichs) scheme
        T .= ?

    elseif FD.Method.Adv == :slf
        # Leapfrog (staggered in time); first step via upwind-like start
        if i == 2
            T .= ?
        else
            T .= ?
        end
        TWE2 .= TWE

    elseif FD.Method.Adv == :semilag
        # Semi-Lagrangian (cubic spline interpolation)
        X          .= ?
        itp_cubic   = cubic_spline_interpolation(xce, TWE)
        T          .= itp_cubic.(X)

    elseif FD.Method.Adv == :tracers
        # Tracer method (NOTE: can be optimized)
        Itp1D_Centers2Markers!(Tm, xm, TWE, xce, Δx, xmin - Δx)
        RK4O1D!(xm, Δt, vx, xmin, xmax)
        Itp1D_Markers2Centers!(T, xc, Tm, xm, Δx, xmin)
    end
    
    display(string("ΔT=", ((Tmax - maximum(T)) / Tmax) * 100))

    # Plot current profile
    q = plot( ? )
    
    if FD.Method.Adv == :tracers 
        plot!(xm, Tm, markershape=:circle, label="", linealpha=:0)
    end

    if save_fig == 1
        Plots.frame(anim)
    else
        display(q)
    end
end

# Save animation -------------------------------------------------------- #
if save_fig == 1
    # Write the frames to a GIF file
    Plots.gif(anim, string(path, filename, ".gif"), fps = 15)
    foreach(rm, filter(startswith(string(path, "00")), readdir(path, join=true)))
end
# ----------------------------------------------------------------------- #
